# 03 · Graph exploration in Neo4j

Sanity queries over the **test-setup** graph (fit period 9–14 Nov) loaded into Neo4j with
only the **pruned** co-click edges (ADR 004).

**Prerequisites**

```bash
colima start --memory 4                    # Colima does not start automatically after a reboot
docker compose up -d                       # Neo4j 5 Community on localhost:7474 / 7687
uv run python -m recsys.graph.neo4j_load   # wipes and reloads the test setup (~15 s)
```

Schema: `(:Article)-[:IN_SUBCATEGORY]->(:Subcategory)-[:IN_CATEGORY]->(:Category)`,
`(:Article)-[:MENTIONS {source, confidence}]->(:Entity)-[:IS_TYPE]->(:EntityType)`,
`(:Article)-[:COCLICKED {weight, norm_weight}]-(:Article)` (stored once, match undirected).

In [1]:
import pandas as pd
from neo4j import Result

from recsys.graph.build import load_graph
from recsys.graph.neo4j_load import connect

pd.set_option("display.max_colwidth", 90)
driver = connect()

def q(cypher: str, **params) -> pd.DataFrame:
    # Run a read query and return the result as a DataFrame.
    return driver.execute_query(cypher, params, result_transformer_=Result.to_df)

## 1. Load check: Neo4j matches the parquet tables

In [2]:
graph = load_graph("test")
expected = {
    "Article": len(graph["nodes_article"]), "Subcategory": len(graph["nodes_subcategory"]),
    "Category": len(graph["nodes_category"]), "Entity": len(graph["nodes_entity"]),
    "EntityType": len(graph["nodes_entity_type"]),
    "IN_SUBCATEGORY": len(graph["edges_article_in_subcategory"]),
    "IN_CATEGORY": len(graph["edges_subcategory_in_category"]),
    "MENTIONS": len(graph["edges_article_mentions_entity"]),
    "IS_TYPE": len(graph["edges_entity_is_type"]),
    "COCLICKED": len(graph["edges_article_coclicked_article_pruned"]),
}
nodes = q("MATCH (n) RETURN labels(n)[0] AS name, count(*) AS neo4j")
rels = q("MATCH ()-[r]->() RETURN type(r) AS name, count(*) AS neo4j")
check = pd.concat([nodes, rels]).set_index("name").assign(parquet=pd.Series(expected))
assert (check["neo4j"] == check["parquet"]).all(), check
check

,neo4j,parquet
name,,
Article,65238,65238
Entity,32472,32472
Category,18,18
Subcategory,291,291
EntityType,6,6
IN_SUBCATEGORY,65238,65238
IN_CATEGORY,291,291
MENTIONS,200650,200650
IS_TYPE,32472,32472


**Interpretation:** Every node and relationship count in Neo4j matches the parquet tables exactly, including the 228,837 pruned co-click edges (the raw 16.1M edges were deliberately not loaded).

## 2. Most-mentioned person entities

Counts distinct articles. The label falls back to MIND's own label for the 171 entities without a Wikidata label.

In [3]:
q('''
MATCH (a:Article)-[:MENTIONS]->(e:Entity)-[:IS_TYPE]->(:EntityType {id: "person"})
RETURN coalesce(e.wikidata_label, e.label) AS person, count(DISTINCT a) AS articles,
       count(DISTINCT a.category) AS categories
ORDER BY articles DESC LIMIT 12
''')

,person,articles,categories
0,Donald Trump,2676,13
1,Joe Biden,259,4
2,Tom Brady,236,5
3,Bill Belichick,174,1
4,Rudy Giuliani,169,3
5,Lamar Jackson,166,2
6,Elizabeth Warren,159,5
7,LeBron James,145,8
8,Elijah Cummings,139,4
9,Tiger Woods,130,3


**Interpretation:** Donald Trump dominates (2,676 articles across 13 categories, ten times the next person), followed by a mix of politics (Biden, Giuliani, Warren) and sports figures (Brady, Belichick, LeBron). Entities like Trump act as hubs that connect almost everything, which is why entity edges need degree-aware weighting.

## 3. Subcategories with the most co-click links to other subcategories

First the subcategory pairs with the most pruned co-click edges between them, then, per
subcategory, the share of its co-click edges that leave the subcategory.

In [4]:
q('''
MATCH (a:Article)-[c:COCLICKED]->(b:Article),
      (a)-[:IN_SUBCATEGORY]->(sa), (b)-[:IN_SUBCATEGORY]->(sb)
WHERE sa <> sb
WITH CASE WHEN sa.id < sb.id THEN [sa.id, sb.id] ELSE [sb.id, sa.id] END AS pair, c
RETURN pair[0] AS subcategory_a, pair[1] AS subcategory_b, count(c) AS links,
       round(avg(c.norm_weight), 3) AS mean_norm_weight
ORDER BY links DESC LIMIT 10
''')

,subcategory_a,subcategory_b,links,mean_norm_weight
0,news/newscrime,news/newsus,4460,0.071
1,news/newspolitics,news/newsus,4077,0.074
2,news/newsus,news/newsworld,3357,0.070
3,news/newspolitics,news/newsworld,3176,0.083
4,news/newsus,sports/football_nfl,1766,0.060
5,news/newscrime,news/newsworld,1745,0.071
6,sports/football_ncaa,sports/football_nfl,1217,0.076
7,sports/baseball_mlb,sports/football_nfl,1207,0.066
8,finance/finance-companies,news/newsus,1173,0.057
9,sports/basketball_nba,sports/football_nfl,1165,0.074


In [5]:
q('''
MATCH (a:Article)-[c:COCLICKED]-(b:Article),
      (a)-[:IN_SUBCATEGORY]->(sa), (b)-[:IN_SUBCATEGORY]->(sb)
WITH sa, count(c) AS edges, sum(CASE WHEN sa <> sb THEN 1 ELSE 0 END) AS cross
WHERE edges >= 2000
RETURN sa.id AS subcategory, edges, cross, round(1.0 * cross / edges, 2) AS cross_share
ORDER BY cross DESC LIMIT 12
''')

,subcategory,edges,cross,cross_share
0,news/newsus,41768,34208,0.82
1,news/newspolitics,36295,22419,0.62
2,news/newsworld,27517,20553,0.75
3,news/newscrime,30714,18712,0.61
4,sports/football_nfl,33232,16958,0.51
5,tv/tv-celebrity,11668,9914,0.85
6,lifestyle/lifestylebuzz,10292,9350,0.91
7,finance/finance-companies,10209,9131,0.89
8,finance/markets,9747,7871,0.81
9,sports/baseball_mlb,11583,6521,0.56


**Interpretation:** The strongest cross-subcategory links are within the news category (crime, politics, US and world news are read together), followed by links within sports. `news/newsus` → `sports/football_nfl` (1,766 links) is the largest cross-category pair. Lifestyle, celebrity, finance and travel subcategories send 81–91% of their co-click edges elsewhere, so their readers browse widely. NFL and MLB keep about half of theirs in-house. Co-click edges therefore add cross-topic signal that subcategory edges alone cannot.

## 4. Leakage check: no co-click edge touches an unseen article

Co-click edges come only from training-period clicks (ADR 002), so articles not seen in
training must have none. This is the property the unit tests check, verified here on the loaded data.

In [6]:
leak = q('''
MATCH (a:Article {seen_in_training: false})
OPTIONAL MATCH (a)-[c:COCLICKED]-()
RETURN count(DISTINCT a) AS articles_unseen_in_training, count(c) AS coclick_edges_on_unseen
''')
assert leak.loc[0, "coclick_edges_on_unseen"] == 0
leak

,articles_unseen_in_training,coclick_edges_on_unseen
0,13956,0


**Interpretation:** None of the 13,956 articles unseen in training (all target-day candidates and history-only articles from the test split) has a co-click edge. The training-period-only rule (ADR 002) holds on the loaded data, not just in the unit tests.

## 5. Nearest neighbours of an unseen test article

An unseen article has no co-click edges, so its neighbourhood comes only from content:
articles sharing entities with it, each shared entity weighted by `1 / log(1 + entity degree)`
so that very common entities (e.g. *United States*) count less, plus a bonus for the same
subcategory. This is a hand-written version of what a GNN will learn to aggregate.

Sample article: the first (by ID) unseen test candidate with 4–8 distinct entities.

In [7]:
sample = q('''
MATCH (a:Article {seen_in_training: false})-[:MENTIONS]->(e)
WITH a, count(DISTINCT e) AS entities WHERE 4 <= entities <= 8
RETURN a.id AS id, a.title AS title, a.category AS category, entities ORDER BY id LIMIT 1
''')
sample_id = sample.loc[0, "id"]
display(sample)
q('''
MATCH (a:Article {id: $id})-[:MENTIONS]->(e:Entity)
WITH DISTINCT e  // an entity can be mentioned in both title and abstract
RETURN coalesce(e.wikidata_label, e.label) AS entity, [(e)-[:IS_TYPE]->(t) | t.id][0] AS type,
       COUNT { (e)<-[:MENTIONS]-() } AS entity_degree
ORDER BY entity_degree DESC
''', id=sample_id)

,id,title,category,entities
0,N1,"Lindsey Graham to Trump: ""I will hold you accountable""",video,5


,entity,type,entity_degree
0,Donald Trump,person,3991
1,Syria,place,574
2,Turkey,place,299
3,Lindsey Graham,person,69
4,United States intervention in Syria,event,20


In [8]:
q('''
MATCH (a:Article {id: $id})-[:MENTIONS]->(e:Entity)<-[:MENTIONS]-(b:Article)
WHERE b <> a
WITH DISTINCT a, b, e  // count each shared entity once, not once per title/abstract mention
WITH a, b, e, COUNT { (e)<-[:MENTIONS]-() } AS degree
WITH a, b, sum(1.0 / log(1 + degree)) AS entity_score,
     collect(DISTINCT coalesce(e.wikidata_label, e.label)) AS shared_entities
OPTIONAL MATCH (a)-[:IN_SUBCATEGORY]->(s)<-[:IN_SUBCATEGORY]-(b)
WITH b, entity_score + CASE WHEN s IS NULL THEN 0 ELSE 0.25 END AS score, shared_entities, s IS NOT NULL AS same_subcategory
RETURN b.id AS id, b.title AS title, b.seen_in_training AS seen, round(score, 3) AS score,
       same_subcategory, shared_entities
ORDER BY score DESC, id LIMIT 10
''', id=sample_id)

,id,title,seen,score,same_subcategory,shared_entities
0,N37706,Trump defends Syria withdrawal amid reports of atrocities and ISIS supporters escaping,True,0.782,False,"[Turkey, United States intervention in Syria, Donald Trump, Syria]"
1,N14216,Trump had 'wonderful' meeting with Erdogan,True,0.703,True,"[Turkey, Donald Trump, Syria]"
2,N15899,"Trump touts friendship with Turkey's Erdogan, says he didn't watch impeachment hearing",True,0.703,True,"[Turkey, Donald Trump, Syria]"
3,N16514,Former Trump national security official on Trump & Syria: 'This is just obscene',True,0.703,True,"[Turkey, Donald Trump, Syria]"
4,N25002,"Trump lifts sanctions on Turkey, claims victory 'against all evidence from the ground'",False,0.703,True,"[Turkey, Donald Trump, Syria]"
5,N4749,Zakaria on Syria: Trump revealing he is a paper tiger,False,0.703,True,"[Turkey, Donald Trump, Syria]"
6,N53149,"Fallout from Trump's change in Syria policy ""essentially irreversible,"" retired admira...",False,0.703,True,"[Turkey, Donald Trump, Syria]"
7,N64773,Evangelicals denounce Trump for first time over Syria,True,0.703,True,"[Turkey, Donald Trump, Syria]"
8,N71,Military and diplomatic fallout from cease-fire in Syria,True,0.703,True,"[Turkey, Donald Trump, Syria]"
9,N1282,Trump doubles down on Syria withdrawal as criticism continues,True,0.606,False,"[United States intervention in Syria, Donald Trump, Syria]"


**Interpretation:** The sample unseen article (Lindsey Graham confronting Trump over Syria) has no co-click edges, yet its content neighbours are all on-topic: Trump's Syria withdrawal, sanctions on Turkey, the Erdogan meeting. Down-weighting by entity degree stops *Donald Trump* (degree 3,991) from making every Trump article a neighbour. Three of the ten neighbours are themselves unseen, so content edges also connect new articles to each other. The +0.25 same-subcategory bonus is large compared with the entity scores here; a learned model will weight these signals itself.

## 6. Entity-type mix by category

In [9]:
mix = q('''
MATCH (a:Article)-[:MENTIONS]->(:Entity)-[:IS_TYPE]->(t:EntityType)
RETURN a.category AS category, t.id AS entity_type, count(*) AS mentions
''')
table = mix.pivot(index="category", columns="entity_type", values="mentions").fillna(0)
table = table[table.sum(axis=1) >= 3000]
shares = table.div(table.sum(axis=1), axis=0)[["person", "place", "organisation", "event", "creative work", "other"]]
shares.sort_values("person", ascending=False).style.format("{:.0%}").background_gradient(axis=None, cmap="Blues")

entity_type,person,place,organisation,event,creative work,other
category,,,,,,
tv,41%,13%,16%,4%,17%,9%
music,40%,22%,16%,6%,7%,10%
sports,34%,12%,44%,4%,2%,3%
video,33%,25%,23%,5%,5%,10%
lifestyle,18%,30%,21%,9%,8%,15%
news,17%,44%,24%,4%,2%,9%
finance,9%,37%,36%,5%,2%,11%
travel,4%,55%,23%,5%,3%,10%
foodanddrink,4%,36%,24%,11%,7%,19%


**Interpretation:** The entity-type mix follows the content: TV, music and sports mention people most (34–41%), sports is the most organisation-heavy (44%, mostly teams), and weather, travel and news are dominated by places (44–66%). Autos has the largest 'other' share (29%), which matches the car models found in the 'other' type review (ADR 005).

## Takeaways

- **The load is faithful**: Neo4j matches the parquet tables one-to-one, and the leakage property holds in the database.
- **Unseen articles are reachable through content alone**: shared entities, weighted by inverse degree, give topical neighbours with no click data. This is the mechanism the graph model has to learn.
- **Hub entities and hub subcategories need degree normalisation** (Trump: about 4k articles; `news/newsus`: 42k co-click edge endpoints) for message passing, as noted in ADR 003.
- **Co-click edges are mostly cross-subcategory** for news, lifestyle and finance, so they add information beyond the subcategory tree.
- **Entity types separate the domains clearly** (people for entertainment and sports, places for news, travel and weather), which supports keeping the entity-type node (ADR 005).

In [10]:
driver.close()